# lab_03 — rozwiązania

## Setup

In [20]:
from dask.distributed import Client
import os

client = Client(n_workers=2, threads_per_worker=2, memory_limit="2GiB")
print(client.dashboard_link)

DATAPATH = os.path.join("..", "data")
os.makedirs(DATAPATH, exist_ok=True)
os.makedirs(os.path.join(DATAPATH, "expired"), exist_ok=True)


C:\Users\mix13\Documents\GitHub\analiza_duzych_zbiorow\.venv\Lib\site-packages\distributed\node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 60844 instead
  warnings.warn(


http://127.0.0.1:60844/status


## Dane people

In [21]:
import json
import dask
import dask.bag as db
import dask.dataframe as dd

b = dask.datasets.make_people(npartitions=40, records_per_partition=5000)
b.map(json.dumps).to_textfiles(os.path.join(DATAPATH, "people_*.json"))


['C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_00.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_01.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_02.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_03.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_04.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_05.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_06.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_07.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_08.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_09.json',
 'C:/Users/mix13/Documents/GitHub/analiza_duzych_zbiorow/lab03/../data/people_10.json',
 'C:/Users/mix13/Documents/GitHu

In [22]:
b = db.read_text(os.path.join(DATAPATH, "people_*.json")).map(json.loads)
b.take(1)


({'age': 1,
  'name': ['Arnulfo', 'Ray'],
  'occupation': 'Booking Clerk',
  'telephone': '+1-317-939-8781',
  'address': {'address': '31 Balhi Bridge', 'city': 'Fairborn'},
  'credit-card': {'number': '4953 0032 8541 1180',
   'expiration-date': '07/16'}},)

## Zadanie 1

In [23]:
from datetime import datetime
import time
import pandas as pd

with open(os.path.join(DATAPATH, "Apache_2k.log"), encoding="utf-8") as f:
    base_lines = f.read().splitlines()

lines = base_lines * 250
print(len(lines))


def parse(inp: str):
    record = {}
    date_start = inp.find("[") + 1
    date_end = inp.find("]")
    level_start = inp.find("[", date_end) + 1
    level_end = inp.find("]", level_start)
    client_start = inp.find("[client ", level_end)
    client_end = inp.find("]", client_start)

    record["date"] = inp[date_start:date_end]
    record["level"] = inp[level_start:level_end]
    record["client"] = "" if client_start == -1 else inp[client_start + 8 : client_end]
    record["message"] = inp[client_end + 2 :] if record["client"] else inp[level_end + 2 :]
    return record


def convert_date(rec):
    rec["date"] = datetime.strptime(rec["date"], "%a %b %d %H:%M:%S %Y")
    return rec


500000


In [24]:
t0 = time.perf_counter()
output = []
for line in lines:
    record = convert_date(parse(line))
    output.append(list(record.values()))
df_seq = pd.DataFrame(output, columns=["date", "level", "client", "message"])
t_seq = time.perf_counter() - t0
print("seq:", t_seq, len(df_seq))
df_seq.head()


seq: 3.0780678000010084 500000


,date,level,client,message
0,2005-12-04 04:47:44,notice,,workerEnv.init() ok /etc/httpd/conf/workers2.p...
1,2005-12-04 04:47:44,error,,mod_jk child workerEnv in error state 6
2,2005-12-04 04:51:08,notice,,jk2_init() Found child 6725 in scoreboard slot 10
3,2005-12-04 04:51:09,notice,,jk2_init() Found child 6726 in scoreboard slot 8
4,2005-12-04 04:51:09,notice,,jk2_init() Found child 6728 in scoreboard slot 6


In [25]:
def process_chunk(chunk):
    rows = []
    for line in chunk:
        record = convert_date(parse(line))
        rows.append(list(record.values()))
    return pd.DataFrame(rows, columns=["date", "level", "client", "message"])


chunk_size = 50_000
chunks = [lines[i : i + chunk_size] for i in range(0, len(lines), chunk_size)]

t0 = time.perf_counter()
parts = [dask.delayed(process_chunk)(c) for c in chunks]
logs_ddf = dd.from_delayed(parts)
logs_ddf.to_parquet(os.path.join(DATAPATH, "apache_parquet"), overwrite=True)
t_par = time.perf_counter() - t0
print("delayed:", t_par, "seq:", t_seq)


delayed: 3.271481000003405 seq: 3.0780678000010084


In [26]:
logs_check = dd.read_parquet(os.path.join(DATAPATH, "apache_parquet"))
print(logs_check.npartitions, len(logs_check))
logs_check.head()


10 500000


,date,level,client,message
0,2005-12-04 04:47:44,notice,,workerEnv.init() ok /etc/httpd/conf/workers2.p...
1,2005-12-04 04:47:44,error,,mod_jk child workerEnv in error state 6
2,2005-12-04 04:51:08,notice,,jk2_init() Found child 6725 in scoreboard slot 10
3,2005-12-04 04:51:09,notice,,jk2_init() Found child 6726 in scoreboard slot 8
4,2005-12-04 04:51:09,notice,,jk2_init() Found child 6728 in scoreboard slot 6


## Zadanie 2

In [27]:
from datetime import datetime

b = db.read_text(os.path.join(DATAPATH, "people_*.json")).map(json.loads)

now = datetime.now()
current_month = datetime(now.year, now.month, 1)

expired = b.filter(
    lambda r: datetime.strptime(r["credit-card"]["expiration-date"], "%m/%y") < current_month
)
expired.take(1)


({'age': 1,
  'name': ['Arnulfo', 'Ray'],
  'occupation': 'Booking Clerk',
  'telephone': '+1-317-939-8781',
  'address': {'address': '31 Balhi Bridge', 'city': 'Fairborn'},
  'credit-card': {'number': '4953 0032 8541 1180',
   'expiration-date': '07/16'}},)

In [28]:
expired.repartition(10).map(json.dumps).to_textfiles(
    os.path.join(DATAPATH, "expired", "expired_*.json")
)
print(sorted(os.listdir(os.path.join(DATAPATH, "expired"))))


['expired_0.json', 'expired_1.json', 'expired_2.json', 'expired_3.json', 'expired_4.json', 'expired_5.json', 'expired_6.json', 'expired_7.json', 'expired_8.json', 'expired_9.json']


## Zadanie 3

In [29]:
adults = b.filter(lambda record: record["age"] >= 18)

def flatten(record):
    return {
        "age": record["age"],
        "first_name": record["name"][0],
        "last_name": record["name"][1],
        "occupation": record["occupation"],
        "telephone": record["telephone"],
        "address": record["address"]["address"],
        "city": record["address"]["city"],
        "credit_card_number": record["credit-card"]["number"],
        "credit_card_expiration": record["credit-card"]["expiration-date"],
    }

adults_df = adults.map(flatten).to_dataframe()
adults_df


,age,first_name,last_name,occupation,telephone,address,city,credit_card_number,credit_card_expiration
npartitions=40,,,,,,,,,
,int64,str,str,str,str,str,str,str,str
,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...


In [30]:
adults_df.repartition(npartitions=1).to_parquet(
    os.path.join(DATAPATH, "adults_parquet"), overwrite=True
)
check = dd.read_parquet(os.path.join(DATAPATH, "adults_parquet"))
print(check.npartitions, len(check))
check.head()


1 170286


,age,first_name,last_name,occupation,telephone,address,city,credit_card_number,credit_card_expiration
0,119,Arlena,Burt,Signalman,+14133903217,1358 Dewey Terrace,Holyoke,4580 7865 1034 4322,12/18
1,94,Tish,Ruiz,Warehousewoman,+12070950204,1090 Prentiss Bayou,Cedar City,3780 409804 49177,11/22
2,31,Vernon,Wong,Rent Collector,+12527261705,685 Britton Boulevard,Elmwood Park,4262 8658 0362 9014,09/23
3,84,Cherlyn,Dudley,Optical,+16672219336,624 Jules Gardens,Clifton,4253 4154 9324 5175,10/25
4,18,Vertie,Cortez,Proprietor,+1-903-086-8034,353 Plymouth Viaduct,Aurora,3450 401904 97438,05/19


## Sekcja 4 (klaster w pracowni)

In [31]:
# lokalnie tylko symulacja scenariusza 1
import dask.array as da

x = da.random.random((8_000, 8_000), chunks=(2000, 2000))
print((x + x.T).mean().compute())


0.9999933709654002


In [32]:
# client.close()